# Submission 4 — LightGBM stacker (leaderboard-performance system, ID-DERIVED)

**Disclosure first.** This is the team's leaderboard-performance model. It is **heavily ID-dependent**.
Its ID-derived inputs are the plain ID prior, the similarity-weighted ID prior (SID), the XLM-R ID-head
probabilities, the labels of the nearest training rows by id, the class shares in id windows and the
raw id. These signals exist only because row ids group comments from the same collection source in
this dataset. They are **not valid production features** for newly arriving comments. The team's genuine
text contribution (TF-IDF, text-kNN, E5, text features, grouped validation, error analysis, text-only
ablation) is reported separately.

**Validation class: STANDARD_OOF_STACKING_WITH_DEPENDENCY.** Every base probability and every
neighbour-label feature is out-of-fold for its own row. However, when the stacker for held-out fold k
is trained, its training rows' features come from base models and neighbour pools that included
fold k's labels (standard stacking). The class-2 correction for fold k is chosen on the other folds'
stacker predictions (one level of nesting, not two). A stricter rebuild was attempted: every base
source refit per fold pair, E5 and the ID-head dropped because they cannot be refit on CPU, neighbour
pools at 60% density. Its first outer fold scored 0.6619 uncorrected against 0.6975 here, so that
variant is not used. See the README.

**Task.** Three-class harmful-content classification (0 explicit, 1 subtle, 2 non-toxic) of Bangla,
English, Banglish and mixed-script comments. The metric is macro-F1.

**Validation.** The five grouped folds (fold set B) are loaded from `artifacts_submission_4/fold_ids.npy`,
never regenerated. Identical cleaned texts always share a fold, and the notebook asserts that zero groups cross folds.

**Runtime.** About 10–15 minutes on a 2-core CPU. No GPU and no internet needed.

In [ ]:
# ============ 1. setup, paths, versions ============
import os, re, json, time, hashlib, platform, warnings
from pathlib import Path
import numpy as np, pandas as pd, scipy.sparse as sp, sklearn, lightgbm as lgb
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import normalize
warnings.filterwarnings("ignore")
T0 = time.time()
HERE = Path.cwd()
ART = next(p for p in [HERE / "artifacts_submission_4", HERE / "reproducible" / "artifacts_submission_4",
                       HERE / "uap_submission" / "reproducible" / "artifacts_submission_4"] if p.is_dir())
CFG = json.loads((ART / "config_submission_4.json").read_text(encoding="utf-8"))


def find_data(name):
    for b in [HERE, HERE.parent, HERE.parent.parent]:
        if (b / name).exists():
            return b / name
    raise FileNotFoundError(f"{name} not found next to the notebook or up to two folders above (cwd {HERE})")


def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as fh:
        for c in iter(lambda: fh.read(1 << 20), b""):
            h.update(c)
    return h.hexdigest()


VERS = dict(python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__,
            sklearn=sklearn.__version__, lightgbm=lgb.__version__)
print("versions:", VERS, "\naudited :", CFG["audited_versions"])

In [ ]:
# ============ 2. data, artifacts, folds: hash-verified ============
TRAIN, TEST, SAMPLE = find_data("train.csv"), find_data("test.csv"), find_data("sample_submission.csv")
for p, k in ((TRAIN, "train_sha256"), (TEST, "test_sha256"), (SAMPLE, "sample_sha256")):
    assert sha256(p) == CFG["data"][k], f"{p.name}: hash mismatch"
for rel, h in CFG["artifacts"].items():
    assert sha256(ART / rel) == h, f"{rel}: hash mismatch"
train, test, sample = pd.read_csv(TRAIN, encoding="utf-8"), pd.read_csv(TEST, encoding="utf-8"), pd.read_csv(SAMPLE)
assert list(train.columns) == ["id", "text", "y"] and list(test.columns) == ["id", "text"]
assert np.array_equal(test["id"].to_numpy(), sample["id"].to_numpy())
y = train["y"].to_numpy(np.int64); ids = train["id"].to_numpy(np.int64); tids = test["id"].to_numpy(np.int64)
folds = np.load(ART / "fold_ids.npy").astype(np.int64)
assert hashlib.sha256(folds.tobytes()).hexdigest() == CFG["fold_int64_sha256"]
assert np.bincount(folds).tolist() == [9543, 9546, 9571, 9581, 9576]
URL = re.compile(r"(?:https?://|www\.)\S+", re.I); MEN = re.compile(r"@\w+"); WS = re.compile(r"\s+", re.UNICODE)
clean = lambda s: WS.sub(" ", MEN.sub(" USER ", URL.sub(" URL ", str(s)))).strip()
ctr = train["text"].astype(str).map(clean).to_numpy(); cte = test["text"].astype(str).map(clean).to_numpy()
g = pd.factorize(pd.Index([c.lower().strip() for c in ctr], dtype="object"))[0]
assert int((pd.DataFrame({"g": g, "f": folds}).groupby("g")["f"].nunique() > 1).sum()) == 0
print("all hashes OK; 0 duplicate groups cross folds")


def mf1(yt, p):
    cm = np.bincount(yt * 3 + p, minlength=9).reshape(3, 3); tp = np.diag(cm).astype(float)
    P, R = tp / np.maximum(cm.sum(0), 1), tp / np.maximum(cm.sum(1), 1)
    return float((2 * P * R / np.maximum(P + R, 1e-12)).mean())


def f1s(yt, p):
    cm = np.bincount(yt * 3 + p, minlength=9).reshape(3, 3); tp = np.diag(cm).astype(float)
    P, R = tp / np.maximum(cm.sum(0), 1), tp / np.maximum(cm.sum(1), 1)
    return 2 * P * R / np.maximum(P + R, 1e-12)

## Base sources (standard 5-fold out-of-fold; test from the full training set)
| source | family | built here? |
|---|---|---|
| `tfidf` word(1,2)+char_wb(3,5) TF-IDF → balanced LogisticRegression | lexical text | yes |
| `knn` soft 50-NN vote in char_wb(2,4) TF-IDF space, weight exp(10·cos) | lexical text | yes |
| `e5` frozen intfloat/multilingual-e5-base + scaled balanced LR | semantic text | bundled GPU array (SHA256) |
| `sid` SID config A (Submission 3) | **ID-derived** | yes |
| `pid` plain ID prior, k=20 each side, 1/(d+1e-6) | **ID-derived** | yes |
| `idh` XLM-R base with ID-prior head | **ID-derived** + text | bundled GPU array (SHA256) |

In [ ]:
# ============ 3. rebuild CPU base sources ============
def tfidf_source():
    wk = dict(analyzer="word", ngram_range=(1, 2), min_df=2, max_features=60000, sublinear_tf=True, dtype=np.float32)
    ck = dict(analyzer="char_wb", ngram_range=(3, 5), min_df=2, max_features=60000, sublinear_tf=True, dtype=np.float32)
    lk = dict(solver="lbfgs", C=4.0, class_weight="balanced", max_iter=2000, random_state=42)
    def feats(fit, *apply):
        w, c = TfidfVectorizer(**wk).fit(fit), TfidfVectorizer(**ck).fit(fit)
        return [sp.hstack([w.transform(a), c.transform(a)], format="csr") for a in apply]
    oof = np.zeros((len(y), 3))
    for k in range(5):
        tr, va = np.flatnonzero(folds != k), np.flatnonzero(folds == k)
        A, B = feats(ctr[tr], ctr[tr], ctr[va]); oof[va] = LogisticRegression(**lk).fit(A, y[tr]).predict_proba(B)
    A, B = feats(ctr, ctr, cte)
    return oof, LogisticRegression(**lk).fit(A, y).predict_proba(B)


def knn_source(K=50, beta=10.0, chunk=2000):
    lo, lt = np.array([s.lower() for s in ctr], dtype=object), np.array([s.lower() for s in cte], dtype=object)
    V = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), min_df=2, sublinear_tf=True, max_features=150000,
                        dtype=np.float32).fit(lo)
    A, AT = normalize(V.transform(lo)).tocsr(), normalize(V.transform(lt)).tocsr()
    def vote(Q, P, yp):
        out = np.zeros((Q.shape[0], 3))
        for s in range(0, Q.shape[0], chunk):
            S = (Q[s:s + chunk] @ P.T).toarray(); idx = np.argpartition(-S, K, axis=1)[:, :K]
            w = np.exp(beta * np.take_along_axis(S, idx, 1)); yy = yp[idx]
            pr = np.stack([(w * (yy == c)).sum(1) for c in range(3)], 1); out[s:s + chunk] = pr / pr.sum(1, keepdims=True)
        return out
    oof = np.zeros((len(y), 3))
    for k in range(5):
        tr, va = np.flatnonzero(folds != k), np.flatnonzero(folds == k); oof[va] = vote(A[va], A[tr], y[tr])
    return oof, vote(AT, A, y)


def sid_source(KW=120, k=180, beta=10.0, gamma=1.0, chunk=800):
    V = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), min_df=2, sublinear_tf=True, max_features=80000,
                        dtype=np.float32).fit(ctr)
    Xn = normalize(V.transform(ctr)).tocsr()
    def sid(q_ids, Xq, pool_rows):
        o = np.argsort(ids[pool_rows], kind="stable"); pid, prow = ids[pool_rows][o], pool_rows[o]
        assert len(np.intersect1d(q_ids, pid)) == 0
        out = np.zeros((len(q_ids), 3))
        for s in range(0, len(q_ids), chunk):
            qi, xq = q_ids[s:s + chunk], Xq[s:s + chunk]; pos = np.searchsorted(pid, qi)
            cand = pos[:, None] + np.arange(-KW, KW)[None, :]; valid = (cand >= 0) & (cand < len(pid))
            cc = np.clip(cand, 0, len(pid) - 1); d = np.abs(pid[cc] - qi[:, None]).astype(float); d[~valid] = np.inf
            order = np.argsort(d, axis=1, kind="stable")[:, :k]; dsel = np.take_along_axis(d, order, 1)
            nrow = prow[np.take_along_axis(cc, order, 1)]; qrep = np.repeat(np.arange(xq.shape[0]), nrow.shape[1])
            sim = np.asarray(xq[qrep].multiply(Xn[nrow.ravel()]).sum(1)).reshape(nrow.shape)
            w = np.where(np.isfinite(dsel), 1.0 / ((np.where(np.isfinite(dsel), dsel, 1.0) + 1e-6) ** gamma), 0.0) * np.exp(beta * sim)
            yy = y[nrow]; P = np.stack([(w * (yy == c)).sum(1) for c in range(3)], 1); tot = P.sum(1, keepdims=True)
            out[s:s + chunk] = np.where(tot > 0, P / np.where(tot > 0, tot, 1), 1 / 3)
        return out
    oof = np.zeros((len(y), 3))
    for f in range(5):
        va, tr = np.flatnonzero(folds == f), np.flatnonzero(folds != f); oof[va] = sid(ids[va], Xn[va], tr)
    return oof, sid(tids, normalize(V.transform(cte)).tocsr(), np.arange(len(y)))


def pid_prior(q, pool_ids, pool_y, k=20, eps=1e-6):
    o = np.argsort(pool_ids, kind="stable"); s, sy = pool_ids[o], pool_y[o]; n = len(s)
    pos = np.searchsorted(s, q); cand = pos[:, None] + np.arange(-k, k + 1)[None, :]
    valid = (cand >= 0) & (cand < n); cc = np.clip(cand, 0, n - 1)
    d = np.abs(s[cc] - q[:, None]).astype(np.float64); d = np.where(valid & (s[cc] != q[:, None]), d, np.inf)
    sel = np.argsort(d, axis=1, kind="stable")[:, :k]; ds = np.take_along_axis(d, sel, 1)
    ys = sy[cc[np.arange(len(q))[:, None], sel]]; good = np.isfinite(ds)
    w = np.where(good, 1.0 / (np.where(good, ds, 0.0) + eps), 0.0)
    P = np.stack([(w * (ys == c)).sum(1) for c in range(3)], 1); tot = P.sum(1, keepdims=True)
    P = P / np.where(tot > 0, tot, 1.0); P[tot.ravel() <= 0] = 1 / 3
    assert int(np.isin(q, s).sum()) == 0
    return P


t0 = time.time()
SRC = {}
SRC["tfidf"] = tfidf_source(); print(f"tfidf {time.time() - t0:.0f}s", flush=True)
SRC["knn"] = knn_source(); print(f"knn {time.time() - t0:.0f}s", flush=True)
z = np.load(ART / "emb_e5-base_final.npz"); SRC["e5"] = (z["oof"], z["test"])
SRC["sid"] = sid_source(); print(f"sid {time.time() - t0:.0f}s", flush=True)
po = np.zeros((len(y), 3))
for k in range(5):
    po[folds == k] = pid_prior(ids[folds == k], ids[folds != k], y[folds != k])
SRC["pid"] = (po, pid_prior(tids, ids, y))
z = np.load(ART / "xlmr_base_idhead_final.npz"); SRC["idh"] = (z["oof"], z["test"])
for n, (o, t) in SRC.items():
    assert o.shape == (len(y), 3) and t.shape == (len(tids), 3) and np.isfinite(o).all() and np.isfinite(t).all()
    print(f"  {n:6s} standalone OOF macro-F1 {mf1(y, o.argmax(1)):.6f}")

## Stacker features (60)
* 18 source probabilities (6 sources × 3 classes).
* **ID-neighbour features (32).** The labels and log id-distances of the 3 nearest training rows on each
  side, plus the count and class shares of training rows within ±3/10/30/100/300 ids. For a row in fold k
  the neighbour pool is the training rows of the **other** folds; a row's own label never enters its features.
  For test rows, the stacker for fold k uses the same pool (folds ≠ k), so the neighbour density matches training.
* 9 text features: length, words, Bangla/Latin script flags, emoji count, capital share, "!", "?", digit share.
* The raw id.

In [ ]:
# ============ 4. features ============
def nb_feats(q, pool_ids, pool_y):
    o = np.argsort(pool_ids); s = pool_ids[o]; sy = pool_y[o]; n = len(s)
    pos = np.searchsorted(s, q); F = {}
    with np.errstate(invalid="ignore", divide="ignore"):
        for i in (1, 2, 3):
            p = pos - i; ok = p >= 0; pc = np.clip(p, 0, n - 1)
            F[f"prev{i}_y"] = np.where(ok, sy[pc], -1); F[f"prev{i}_d"] = np.where(ok, np.log1p(q - s[pc]), 9.0)
            nx = pos + i - 1; ok = nx < n; nc = np.clip(nx, 0, n - 1)
            F[f"next{i}_y"] = np.where(ok, sy[nc], -1); F[f"next{i}_d"] = np.where(ok, np.log1p(s[nc] - q), 9.0)
    cum = np.vstack([np.zeros((1, 3)), np.cumsum(np.eye(3)[sy], 0)])
    for Wn in (3, 10, 30, 100, 300):
        a = np.searchsorted(s, q - Wn, "left"); b = np.searchsorted(s, q + Wn, "right"); c = cum[b] - cum[a]; t = c.sum(1)
        F[f"w{Wn}_n"] = t
        for k in range(3):
            F[f"w{Wn}_p{k}"] = np.where(t > 0, c[:, k] / np.maximum(t, 1), np.nan)
    return pd.DataFrame(F)


def text_meta(txt):
    t = pd.Series(txt).astype(str); L = t.str.len()
    return pd.DataFrame({"len": L, "nwords": t.str.split().str.len(), "bn": t.str.contains("[ঀ-৿]").astype(int),
                         "lat": t.str.contains("[A-Za-z]").astype(int), "emoji": t.str.count("[\U0001F300-\U0001FAFF☀-➿]"),
                         "caps": t.str.count("[A-Z]") / L.clip(lower=1), "excl": t.str.count("!"), "q": t.str.count(r"\?"),
                         "digits": t.str.count(r"\d") / L.clip(lower=1)})


def probs(split):
    return pd.DataFrame({f"{n}_p{k}": SRC[n][0 if split == "tr" else 1][:, k] for n in SRC for k in range(3)})


parts = []
for k in range(5):
    va = np.flatnonzero(folds == k); pool = np.flatnonzero(folds != k)
    f = nb_feats(ids[va], ids[pool], y[pool]); f.index = va; parts.append(f)
nbtr = pd.concat(parts).sort_index()
Ftr = pd.concat([probs("tr"), nbtr.reset_index(drop=True), text_meta(train.text)], axis=1); Ftr["rid"] = ids
Fte0 = pd.concat([probs("te"), nb_feats(tids, ids, y), text_meta(test.text)], axis=1); Fte0["rid"] = tids
assert Ftr.shape == (47817, 60) and Fte0.shape == (11955, 60) and list(Ftr.columns) == CFG["feature_order"]
nbcols = [c for c in Ftr.columns if c.startswith(("prev", "next", "w"))]

## LightGBM stacker
The configuration is fixed and there is no early stopping. The seeds (42, 7, 2026) were declared in
advance, and all three are retained and averaged. Class weights are "balanced", computed from the class counts of all training labels.

In [ ]:
# ============ 5. cross-fitted stacker, 3 seeds ============
cw = len(y) / (3 * np.bincount(y)); wts = cw[y]
OOF = np.zeros((len(y), 3)); TST = np.zeros((len(tids), 3)); SEED_SCORES = {}
for SEED in CFG["seeds"]:
    params = dict(CFG["lightgbm"], seed=SEED, bagging_seed=SEED, feature_fraction_seed=SEED)
    oof = np.zeros((len(y), 3)); tst = np.zeros((len(tids), 3))
    for k in range(5):
        trm, va = folds != k, folds == k
        m = lgb.train(params, lgb.Dataset(Ftr.loc[trm], y[trm], weight=wts[trm]), CFG["rounds"])
        oof[va] = m.predict(Ftr.loc[va])
        pool = np.flatnonzero(folds != k); Fte = Fte0.copy()
        Fte[nbcols] = nb_feats(tids, ids[pool], y[pool])[nbcols].values
        tst += m.predict(Fte) / 5
    p = oof.argmax(1)
    SEED_SCORES[SEED] = dict(macro_f1=round(mf1(y, p), 6), folds=[round(mf1(y[folds == k], p[folds == k]), 4) for k in range(5)])
    print(f"seed {SEED}: {SEED_SCORES[SEED]}  ({time.time() - T0:.0f}s)", flush=True)
    OOF += oof / len(CFG["seeds"]); TST += tst / len(CFG["seeds"])
print("3-seed mean, no correction:", round(mf1(y, OOF.argmax(1)), 6))

## Class-2 correction (one level of nesting) and final metrics
For each held-out fold k, the per-class log-offset `[0, b1, b2]` is chosen on the other four folds'
stacker predictions (grid fixed in advance: b1 ∈ [−0.5, 0.5], b2 ∈ [−1.2, 0.2], step 0.05) and applied to
fold k. For test, the median of the five fold offsets is used. The correction mainly removes the
over-prediction of class 2 caused by the balanced class weights.

In [ ]:
# ============ 6. correction, metrics ============
L = np.log(np.clip(OOF, 1e-9, 1))
GRID = [np.array([0, a, b]) for a in np.arange(-.5, .51, .05) for b in np.arange(-1.2, .21, .05)]
pred = np.zeros(len(y), int); OFFS = []
for k in range(5):
    t = folds != k
    b = max(GRID, key=lambda b: mf1(y[t], (L[t] + b).argmax(1)))
    pred[folds == k] = (L[folds == k] + b).argmax(1); OFFS.append(b)
FINAL_OFF = np.median(np.array(OFFS), 0)
fs = [mf1(y[folds == k], pred[folds == k]) for k in range(5)]
RES = dict(oof_macro_f1=round(mf1(y, pred), 6), per_class=f1s(y, pred).round(6).tolist(),
           accuracy_not_macro_f1=round(float((pred == y).mean()), 6),
           confusion=np.bincount(y * 3 + pred, minlength=9).reshape(3, 3).tolist(), fold_scores=np.round(fs, 6).tolist(),
           fold_std=round(float(np.std(fs)), 6), worst_fold=round(float(min(fs)), 6),
           fold_offsets=[o.round(2).tolist() for o in OFFS], final_offset=FINAL_OFF.round(3).tolist(), seeds=SEED_SCORES)
print(json.dumps(RES, indent=1))
assert abs(RES["oof_macro_f1"] - CFG["audited_oof"]) <= CFG["oof_tolerance"], "OOF drifted from the audited value"
print(f"OOF reproduces the audited {CFG['audited_oof']} within {CFG['oof_tolerance']}")

In [ ]:
# ============ 7. test prediction, duplicate override, CSV ============
tp = (np.log(np.clip(TST, 1e-9, 1)) + FINAL_OFF).argmax(1)
share = np.bincount(tp, minlength=3) / len(tp); tshare = np.bincount(y, minlength=3) / len(y)
print("test class share", share.round(4).tolist(), "train", tshare.round(4).tolist())
assert (np.abs(share - tshare) <= 0.06).all()
gg = pd.DataFrame({"t": pd.Series(list(ctr), dtype="object"), "y": y}).groupby("t")["y"].agg(["nunique", "first"])
LOOKUP = {str(t): int(v) for t, v in gg[gg["nunique"] == 1]["first"].items()}
final = tp.copy(); hits = 0
for i, t in enumerate(cte):
    v = LOOKUP.get(str(t))
    if v is not None:
        final[i] = v; hits += 1
print(f"duplicate override: {hits} matched, {int((final != tp).sum())} changed, {int((gg['nunique'] > 1).sum())} conflicting groups skipped")


def validate(df):
    assert list(df.columns) == ["id", "y_pred"] and len(df) == 11955 and df["id"].is_unique
    assert np.array_equal(df["id"].to_numpy(), sample["id"].to_numpy()) and not df.isna().any().any()
    assert df["y_pred"].dtype.kind in "iu" and set(df["y_pred"].unique()) <= {0, 1, 2}


sub = pd.DataFrame({"id": tids, "y_pred": final.astype(int)}); validate(sub); print("PRE-WRITE VALIDATION OK")
OUT = next((b / "uap_submission" / "submission" for b in [HERE] + list(HERE.parents)
            if (b / "uap_submission" / "submission").is_dir()), HERE / "uap_submission" / "submission")
OUT.mkdir(parents=True, exist_ok=True); CSV = OUT / "submission_4.csv"
sub.to_csv(CSV, index=False); validate(pd.read_csv(CSV)); print("POST-READ VALIDATION OK ->", CSV)
print("CSV SHA256:", sha256(CSV), "| audited:", CFG["audited_csv_sha256"],
      "| byte-identical:", sha256(CSV) == CFG["audited_csv_sha256"])
print(f"total runtime {time.time() - T0:.0f}s   KAGGLE UPLOAD: NO")

## Limitations and rules
* **ID dependence.** Most of the gain over the text-only pipeline (honest text-only OOF about 0.565) comes
  from ID-derived inputs. Treat it as a leaderboard system, not a deployable classifier.
* **Standard stacking dependency** and a one-level-nested correction (see the top of the notebook).
* Label noise (about 11% of duplicate-text groups conflict) and dataset-specific source blocks.
* E5 and the XLM-R ID-head are bundled probability arrays from a Colab T4 run. Retraining them uses fp16
  and is not bit-reproducible, so Mode A uses the SHA256-verified arrays.
* Scores can move by about 1e-3 across scikit-learn/LightGBM versions. The audit versions are printed above.
* No public-leaderboard feedback and no external labelled data were used. Pretrained weights are public checkpoints only.